# SycoLex — Dual-Branch (TRUE vs FLIP) HierarchicalLegalBERT

This notebook incorporates **Priority 1 only**: encoding the TRUE pair and FLIP pair as two
separate chunk sequences, pooling each independently through the shared BERT + attention
mechanism, and feeding the classifier `[true_emb; flip_emb; true_emb - flip_emb; true_emb * flip_emb]`.

Everything else (backbone = InLegalBERT, no-facts instruction extraction, front-truncation of
responses, GroupShuffleSplit, single-linear attention pooling, training loop) is kept identical
to the prior run so that any F1 change can be attributed specifically to this architectural change.


In [1]:
import os
import re
import json
import random
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.cuda.amp import autocast, GradScaler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


In [2]:
MODEL_NAME = "law-ai/InLegalBERT"

MAX_SEQ_LEN = 512
SPECIAL_TOKENS = 2
MAX_CONTENT_LEN = MAX_SEQ_LEN - SPECIAL_TOKENS   # 510
OVERLAP = 128
STRIDE = MAX_CONTENT_LEN - OVERLAP               # 382

# NOTE: previously MAX_CHUNKS=12 applied to the *combined* TRUE+FLIP string.
# Now each side is chunked independently, so we set a per-side cap.
# 8 chunks/side keeps total chunk budget (16) comparable to before, while
# guaranteeing the FLIP side is never starved of chunks by the TRUE side
# consuming the whole budget (a real data-loss issue in the old design).
MAX_CHUNKS_PER_SIDE = 8
print("MAX_CHUNKS_PER_SIDE set to:", MAX_CHUNKS_PER_SIDE)

BATCH_SIZE = 4          # may need to drop to 2 (with GRAD_ACCUM_STEPS=8) if OOM,
                         # since dual-branch forward pass roughly doubles chunk-encoding cost
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 10
EARLY_STOPPING = 5

MODEL_SAVE_PATH = r"C:\Users\bssru\Documents\SycoLex\checkpoints\best_inlegalbert_dualbranch.pt"
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)


MAX_CHUNKS_PER_SIDE set to: 8


In [3]:
DATASET_PATH = r"C:\Users\bssru\Documents\SycoLex\sycolex_merged_dataset_ALL_MODELS.json"

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print("Total Examples:", len(df))

def clean_legal_text(text):
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\u00A0", " ")
    text = re.sub(r"[\u200B-\u200D\uFEFF]", "", text)
    text = text.replace("\r", "\n")
    text = text.replace("\t", " ")
    text = re.sub(r"[ ]+", " ", text)
    text = re.sub(r"\n+", "\n", text)
    return text.strip()

TEXT_COLUMNS = ["fact", "true_prompt", "true_response", "flip_prompt", "flip_response"]
for column in TEXT_COLUMNS:
    df[column] = df[column].apply(clean_legal_text)

print("Text preprocessing completed.")


Total Examples: 10620
Text preprocessing completed.


In [4]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(df, groups=df["case_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

print("Training Samples   :", len(train_df))
print("Validation Samples :", len(val_df))

train_cases = set(train_df["case_id"])
val_cases = set(val_df["case_id"])
print("Leakage check — common cases:", len(train_cases.intersection(val_cases)))


Training Samples   : 8472
Validation Samples : 2148
Leakage check — common cases: 0


In [5]:
def extract_instruction(prompt, fact):
    prompt = str(prompt)
    fact = str(fact)
    instruction = prompt.replace(fact, "")
    instruction = re.sub(r"\n+", "\n", instruction)
    instruction = re.sub(r"[ ]+", " ", instruction)
    return instruction.strip()

for d in [train_df, val_df]:
    d["true_instruction"] = d.apply(lambda row: extract_instruction(row["true_prompt"], row["fact"]), axis=1)
    d["flip_instruction"] = d.apply(lambda row: extract_instruction(row["flip_prompt"], row["fact"]), axis=1)

print("Instruction extraction completed.")


Instruction extraction completed.


In [6]:
COLUMNS_TO_KEEP = [
    "case_id",
    "true_instruction",
    "true_response",
    "flip_instruction",
    "flip_response",
    "prompt_variant",
    "category",
    "label",
    "jurisdiction",
    "model",
]

train_df = train_df[COLUMNS_TO_KEEP].reset_index(drop=True)
val_df = val_df[COLUMNS_TO_KEEP].reset_index(drop=True)

print("Columns kept:", train_df.columns.tolist())


Columns kept: ['case_id', 'true_instruction', 'true_response', 'flip_instruction', 'flip_response', 'prompt_variant', 'category', 'label', 'jurisdiction', 'model']


In [7]:
def build_side_input(instruction, response, prompt_variant, category, side_label):
    """Build a per-side (TRUE-only or FLIP-only) input string.
    Unlike the old build_final_input_no_facts, this does NOT concatenate
    both sides together — each side is encoded independently so the model
    can compare them explicitly rather than implicitly."""
    text = f"""[{side_label} PAIR]

Instruction:

{instruction}

Response:

{response}

Prompt Variant:

{prompt_variant}

Category:

{category}
"""
    return text.strip()

for d in [train_df, val_df]:
    d["true_input"] = d.apply(
        lambda row: build_side_input(row["true_instruction"], row["true_response"],
                                      row["prompt_variant"], row["category"], "TRUE"),
        axis=1
    )
    d["flip_input"] = d.apply(
        lambda row: build_side_input(row["flip_instruction"], row["flip_response"],
                                      row["prompt_variant"], row["category"], "FLIP"),
        axis=1
    )

print("Per-side (true_input / flip_input) fields created.")

tokenizer_check = AutoTokenizer.from_pretrained(MODEL_NAME)
for col in ["true_input", "flip_input"]:
    train_df[col + "_tokens"] = train_df[col].apply(lambda x: len(tokenizer_check.tokenize(str(x))))
    print(f"\n{col} token length stats:")
    print(train_df[col + "_tokens"].describe())


Per-side (true_input / flip_input) fields created.


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2608 > 512). Running this sequence through the model will result in indexing errors



true_input token length stats:
count     8472.000000
mean      2966.030217
std       3224.144998
min        124.000000
25%        966.750000
50%       2424.000000
75%       3452.000000
max      32850.000000
Name: true_input_tokens, dtype: float64

flip_input token length stats:
count     8472.000000
mean      3007.379485
std       3256.883310
min         97.000000
25%        969.000000
50%       2453.500000
75%       3507.250000
max      32014.000000
Name: flip_input_tokens, dtype: float64


In [8]:
def coverage_tokens(n_chunks):
    return n_chunks * STRIDE + MAX_CONTENT_LEN

chunk_options = [4, 6, 8, 10, 12, 16]
for col in ["true_input", "flip_input"]:
    print(f"--- Coverage for {col} ---")
    results = []
    for n in chunk_options:
        cov = coverage_tokens(n)
        pct_covered = (train_df[col + "_tokens"] <= cov).mean() * 100
        results.append({"max_chunks": n, "token_coverage": cov, "pct_docs_fully_covered": round(pct_covered, 2)})
    print(pd.DataFrame(results).to_string(index=False))
    print()


--- Coverage for true_input ---
 max_chunks  token_coverage  pct_docs_fully_covered
          4            2038                   38.90
          6            2802                   61.05
          8            3566                   76.76
         10            4330                   84.57
         12            5094                   88.81
         16            6622                   93.59

--- Coverage for flip_input ---
 max_chunks  token_coverage  pct_docs_fully_covered
          4            2038                   38.70
          6            2802                   60.46
          8            3566                   75.70
         10            4330                   83.68
         12            5094                   88.09
         16            6622                   93.20



In [9]:
def truncate_response(text, tokenizer, max_tokens=900):
    """Cap an individual response so extreme outliers don't blow past the chunk window.
    NOTE: still front-truncation (kept identical to prior run per your request to change
    only the dual-branch architecture). Head+tail truncation is a separate, independent
    improvement not included here."""
    token_ids = tokenizer.encode(str(text), add_special_tokens=False)
    if len(token_ids) <= max_tokens:
        return text
    truncated_ids = token_ids[:max_tokens]
    return tokenizer.decode(truncated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

for d in [train_df, val_df]:
    d["true_response"] = d["true_response"].apply(lambda x: truncate_response(x, tokenizer_check))
    d["flip_response"] = d["flip_response"].apply(lambda x: truncate_response(x, tokenizer_check))

    # rebuild side inputs with capped responses
    d["true_input"] = d.apply(
        lambda row: build_side_input(row["true_instruction"], row["true_response"],
                                      row["prompt_variant"], row["category"], "TRUE"),
        axis=1
    )
    d["flip_input"] = d.apply(
        lambda row: build_side_input(row["flip_instruction"], row["flip_response"],
                                      row["prompt_variant"], row["category"], "FLIP"),
        axis=1
    )

train_df["true_input_tokens"] = train_df["true_input"].apply(lambda x: len(tokenizer_check.tokenize(str(x))))
train_df["flip_input_tokens"] = train_df["flip_input"].apply(lambda x: len(tokenizer_check.tokenize(str(x))))

for n in [6, 8, 10, 12]:
    cov = coverage_tokens(n)
    pct_true = (train_df["true_input_tokens"] <= cov).mean() * 100
    pct_flip = (train_df["flip_input_tokens"] <= cov).mean() * 100
    print(f"MAX_CHUNKS_PER_SIDE={n} (covers {cov} tokens): TRUE {pct_true:.2f}% | FLIP {pct_flip:.2f}% fully covered")


MAX_CHUNKS_PER_SIDE=6 (covers 2802 tokens): TRUE 91.64% | FLIP 91.62% fully covered
MAX_CHUNKS_PER_SIDE=8 (covers 3566 tokens): TRUE 92.50% | FLIP 92.52% fully covered
MAX_CHUNKS_PER_SIDE=10 (covers 4330 tokens): TRUE 93.56% | FLIP 93.61% fully covered
MAX_CHUNKS_PER_SIDE=12 (covers 5094 tokens): TRUE 94.64% | FLIP 94.66% fully covered


In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def create_chunks(text, tokenizer, max_chunks=MAX_CHUNKS_PER_SIDE):
    token_ids = tokenizer.encode(str(text), add_special_tokens=False)
    chunks = []
    for start in range(0, len(token_ids), STRIDE):
        end = start + MAX_CONTENT_LEN
        chunk_ids = token_ids[start:end]
        chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        chunks.append(chunk_text)
        if end >= len(token_ids):
            break
        if len(chunks) >= max_chunks:
            break
    return chunks

print("Tokenizer loaded:", MODEL_NAME)
print("MAX_CHUNKS_PER_SIDE set to:", MAX_CHUNKS_PER_SIDE)


Tokenizer loaded: law-ai/InLegalBERT
MAX_CHUNKS_PER_SIDE set to: 8


In [11]:
class SycoLexDualBranchDataset(Dataset):
    def __init__(self, dataframe, tokenizer):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def _encode_side(self, text):
        text_chunks = create_chunks(text, self.tokenizer)
        input_ids, attention_masks = [], []
        for chunk in text_chunks:
            encoded = self.tokenizer(
                chunk, add_special_tokens=True, max_length=512,
                truncation=True, padding="max_length",
                return_attention_mask=True, return_tensors="pt"
            )
            input_ids.append(encoded["input_ids"].squeeze(0))
            attention_masks.append(encoded["attention_mask"].squeeze(0))
        input_ids = torch.stack(input_ids)
        attention_masks = torch.stack(attention_masks)
        return input_ids, attention_masks

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        label = int(row["label"])

        true_ids, true_masks = self._encode_side(row["true_input"])
        flip_ids, flip_masks = self._encode_side(row["flip_input"])

        return {
            "true_input_ids": true_ids,
            "true_attention_mask": true_masks,
            "flip_input_ids": flip_ids,
            "flip_attention_mask": flip_masks,
            "label": torch.tensor(label, dtype=torch.long),
        }


def _pad_side(id_list, mask_list):
    max_chunks = max(t.shape[0] for t in id_list)
    padded_ids, padded_masks, chunk_masks = [], [], []
    for ids, masks in zip(id_list, mask_list):
        num_chunks = ids.shape[0]
        pad_chunks = max_chunks - num_chunks
        if pad_chunks > 0:
            ids = F.pad(ids, (0, 0, 0, pad_chunks), value=0)
            masks = F.pad(masks, (0, 0, 0, pad_chunks), value=0)
        padded_ids.append(ids)
        padded_masks.append(masks)
        chunk_masks.append(torch.cat([torch.ones(num_chunks), torch.zeros(pad_chunks)]))
    return torch.stack(padded_ids), torch.stack(padded_masks), torch.stack(chunk_masks)


def collate_fn(batch):
    true_ids_list = [item["true_input_ids"] for item in batch]
    true_masks_list = [item["true_attention_mask"] for item in batch]
    flip_ids_list = [item["flip_input_ids"] for item in batch]
    flip_masks_list = [item["flip_attention_mask"] for item in batch]
    labels = torch.stack([item["label"] for item in batch])

    true_input_ids, true_attention_mask, true_chunk_mask = _pad_side(true_ids_list, true_masks_list)
    flip_input_ids, flip_attention_mask, flip_chunk_mask = _pad_side(flip_ids_list, flip_masks_list)

    return {
        "true_input_ids": true_input_ids,
        "true_attention_mask": true_attention_mask,
        "true_chunk_mask": true_chunk_mask,
        "flip_input_ids": flip_input_ids,
        "flip_attention_mask": flip_attention_mask,
        "flip_chunk_mask": flip_chunk_mask,
        "label": labels,
    }


train_dataset = SycoLexDualBranchDataset(train_df, tokenizer)
val_dataset = SycoLexDualBranchDataset(val_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print("Train Batches:", len(train_loader))
print("Validation Batches:", len(val_loader))


Train Batches: 2118
Validation Batches: 537


In [12]:
class DualBranchHierarchicalLegalBERT(nn.Module):
    """Shared BERT + attention-pooling encoder applied independently to the
    TRUE side and the FLIP side. The classifier then sees
    [true_emb; flip_emb; true_emb - flip_emb; true_emb * flip_emb],
    making the true-vs-flip comparison an explicit input feature rather
    than something the model has to reconstruct from a single pooled
    vector over a concatenated blob."""

    def __init__(self, num_classes=2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.bert.gradient_checkpointing_enable()
        hidden_size = self.bert.config.hidden_size
        print("Hidden Size:", hidden_size)

        # shared attention pooling head, used for both branches
        self.attention = nn.Linear(hidden_size, 1)
        self.dropout = nn.Dropout(0.3)

        # classifier input = concat + diff + product => 4x hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 4, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_size, num_classes),
        )

    def _encode_branch(self, input_ids, attention_mask, chunk_mask):
        batch_size, num_chunks, seq_len = input_ids.shape

        flat_input_ids = input_ids.view(batch_size * num_chunks, seq_len)
        flat_attention_mask = attention_mask.view(batch_size * num_chunks, seq_len)

        outputs = self.bert(input_ids=flat_input_ids, attention_mask=flat_attention_mask)

        cls_embeddings = outputs.last_hidden_state[:, 0]
        cls_embeddings = cls_embeddings.view(batch_size, num_chunks, -1)

        attention_scores = self.attention(cls_embeddings).squeeze(-1)
        fill_value = torch.finfo(attention_scores.dtype).min
        attention_scores = attention_scores.masked_fill(chunk_mask == 0, fill_value)
        attention_weights = torch.softmax(attention_scores, dim=1)

        doc_embedding = torch.sum(cls_embeddings * attention_weights.unsqueeze(-1), dim=1)
        return doc_embedding

    def forward(self, true_input_ids, true_attention_mask, true_chunk_mask,
                flip_input_ids, flip_attention_mask, flip_chunk_mask):

        true_emb = self._encode_branch(true_input_ids, true_attention_mask, true_chunk_mask)
        flip_emb = self._encode_branch(flip_input_ids, flip_attention_mask, flip_chunk_mask)

        diff = true_emb - flip_emb
        prod = true_emb * flip_emb

        combined = torch.cat([true_emb, flip_emb, diff, prod], dim=-1)
        combined = self.dropout(combined)

        return self.classifier(combined)


model = DualBranchHierarchicalLegalBERT(num_classes=2)
model = model.to(DEVICE)
print("Model loaded successfully.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hidden Size: 768
Model loaded successfully.


In [13]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

total_training_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_training_steps),
    num_training_steps=total_training_steps
)

scaler = GradScaler()

print("Learning Rate :", LEARNING_RATE)
print("Epochs        :", EPOCHS)
print("Training Steps:", total_training_steps)


Learning Rate : 2e-05
Epochs        : 10
Training Steps: 21180


In [14]:
import gc

model.train()
batch = next(iter(train_loader))

true_input_ids = batch["true_input_ids"].to(DEVICE)
true_attention_mask = batch["true_attention_mask"].to(DEVICE)
true_chunk_mask = batch["true_chunk_mask"].to(DEVICE)
flip_input_ids = batch["flip_input_ids"].to(DEVICE)
flip_attention_mask = batch["flip_attention_mask"].to(DEVICE)
flip_chunk_mask = batch["flip_chunk_mask"].to(DEVICE)
labels = batch["label"].to(DEVICE)

print("TRUE batch shape:", true_input_ids.shape)
print("FLIP batch shape:", flip_input_ids.shape)

oom_occurred = False
try:
    with autocast():
        logits = model(
            true_input_ids, true_attention_mask, true_chunk_mask,
            flip_input_ids, flip_attention_mask, flip_chunk_mask
        )
        loss = criterion(logits, labels)
    scaler.scale(loss).backward()
    optimizer.zero_grad()

    print("Forward + backward pass succeeded.")
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    print(f"GPU Memory Reserved : {torch.cuda.memory_reserved()/1024**3:.2f} GB")
except torch.cuda.OutOfMemoryError:
    oom_occurred = True
    print(f"OOM at BATCH_SIZE={BATCH_SIZE}, MAX_CHUNKS_PER_SIDE={MAX_CHUNKS_PER_SIDE}")
    print("Fix: set BATCH_SIZE=2, GRAD_ACCUM_STEPS=8 (keeps effective batch size at 16),")
    print("     or drop MAX_CHUNKS_PER_SIDE to 6, then re-run from the DataLoader cell onward.")

gc.collect()
torch.cuda.empty_cache()

# re-instantiate fresh before real training (this test step touched gradients)
if not oom_occurred:
    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

    model = DualBranchHierarchicalLegalBERT(num_classes=2).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_training_steps),
        num_training_steps=total_training_steps
    )
    scaler = GradScaler()
    print("\nModel, optimizer, and scheduler re-initialized fresh for real training.")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (646 > 512). Running this sequence through the model will result in indexing errors
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


TRUE batch shape: torch.Size([4, 3, 512])
FLIP batch shape: torch.Size([4, 3, 512])
Forward + backward pass succeeded.
GPU Memory Allocated: 0.43 GB
GPU Memory Reserved : 1.43 GB


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hidden Size: 768

Model, optimizer, and scheduler re-initialized fresh for real training.


In [15]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    predictions, true_labels = [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validation", leave=False, mininterval=5.0, miniters=50):
            true_input_ids = batch["true_input_ids"].to(device)
            true_attention_mask = batch["true_attention_mask"].to(device)
            true_chunk_mask = batch["true_chunk_mask"].to(device)
            flip_input_ids = batch["flip_input_ids"].to(device)
            flip_attention_mask = batch["flip_attention_mask"].to(device)
            flip_chunk_mask = batch["flip_chunk_mask"].to(device)
            labels = batch["label"].to(device)

            with autocast():
                logits = model(
                    true_input_ids, true_attention_mask, true_chunk_mask,
                    flip_input_ids, flip_attention_mask, flip_chunk_mask
                )
                loss = criterion(logits, labels)

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions, zero_division=0)
    recall = recall_score(true_labels, predictions, zero_division=0)
    f1 = f1_score(true_labels, predictions, zero_division=0)

    return avg_loss, accuracy, precision, recall, f1


def train_one_epoch(model, dataloader, criterion, optimizer, scheduler, device, scaler, grad_accum_steps=1):
    model.train()
    total_loss = 0.0
    predictions, true_labels = [], []

    optimizer.zero_grad()

    for batch_idx, batch in enumerate(tqdm(dataloader, desc="Training", leave=False, mininterval=5.0, miniters=50)):
        true_input_ids = batch["true_input_ids"].to(device)
        true_attention_mask = batch["true_attention_mask"].to(device)
        true_chunk_mask = batch["true_chunk_mask"].to(device)
        flip_input_ids = batch["flip_input_ids"].to(device)
        flip_attention_mask = batch["flip_attention_mask"].to(device)
        flip_chunk_mask = batch["flip_chunk_mask"].to(device)
        labels = batch["label"].to(device)

        with autocast():
            logits = model(
                true_input_ids, true_attention_mask, true_chunk_mask,
                flip_input_ids, flip_attention_mask, flip_chunk_mask
            )
            loss = criterion(logits, labels) / grad_accum_steps

        scaler.scale(loss).backward()

        if (batch_idx + 1) % grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * grad_accum_steps
        preds = torch.argmax(logits, dim=1)
        predictions.extend(preds.detach().cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions, zero_division=0)
    recall = recall_score(true_labels, predictions, zero_division=0)
    f1 = f1_score(true_labels, predictions, zero_division=0)

    return avg_loss, accuracy, precision, recall, f1


In [ ]:
HISTORY_CSV_PATH = r"C:\Users\bssru\Documents\SycoLex\checkpoints\training_history_dualbranch.csv"

best_f1 = 0.0
patience = 0
history = []

for epoch in range(EPOCHS):
    print("=" * 70, flush=True)
    print(f"Epoch {epoch + 1}/{EPOCHS}", flush=True)
    print("=" * 70, flush=True)

    train_loss, train_acc, train_prec, train_rec, train_f1 = train_one_epoch(
        model, train_loader, criterion, optimizer, scheduler, DEVICE, scaler,
        grad_accum_steps=GRAD_ACCUM_STEPS
    )

    val_loss, val_acc, val_prec, val_rec, val_f1 = evaluate(
        model, val_loader, criterion, DEVICE
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss, "train_accuracy": train_acc,
        "train_precision": train_prec, "train_recall": train_rec, "train_f1": train_f1,
        "val_loss": val_loss, "val_accuracy": val_acc,
        "val_precision": val_prec, "val_recall": val_rec, "val_f1": val_f1
    })

    print(
        f"\nTRAIN -- Loss: {train_loss:.4f}  Accuracy: {train_acc:.4f}  "
        f"Precision: {train_prec:.4f}  Recall: {train_rec:.4f}  F1: {train_f1:.4f}",
        flush=True
    )
    print(
        f"VAL   -- Loss: {val_loss:.4f}  Accuracy: {val_acc:.4f}  "
        f"Precision: {val_prec:.4f}  Recall: {val_rec:.4f}  F1: {val_f1:.4f}\n",
        flush=True
    )

    history_df = pd.DataFrame(history)
    history_df.to_csv(HISTORY_CSV_PATH, index=False)

    if val_f1 > best_f1:
        best_f1 = val_f1
        patience = 0
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"Best model saved! (Validation F1 = {best_f1:.4f})", flush=True)
    else:
        patience += 1
        print(f"No improvement. Early Stopping Counter: {patience}/{EARLY_STOPPING}", flush=True)

    if patience >= EARLY_STOPPING:
        print("\nEarly stopping triggered.", flush=True)
        break

print("=" * 70, flush=True)
print("Training Finished.", flush=True)
print(f"Best Validation F1: {best_f1:.4f}", flush=True)


Epoch 1/10


Training:   0%|          | 0/2118 [00:00<?, ?it/s]

Validation:   0%|          | 0/537 [00:00<?, ?it/s]


TRAIN -- Loss: 0.6643  Accuracy: 0.5820  Precision: 0.5232  Recall: 0.4198  F1: 0.4658
VAL   -- Loss: 0.5521  Accuracy: 0.7258  Precision: 0.7794  Recall: 0.5513  F1: 0.6458

Best model saved! (Validation F1 = 0.6458)
Epoch 2/10


Training:   0%|          | 0/2118 [00:00<?, ?it/s]


TRAIN -- Loss: 0.5072  Accuracy: 0.7585  Precision: 0.7624  Recall: 0.6446  F1: 0.6986
VAL   -- Loss: 0.4977  Accuracy: 0.7621  Precision: 0.8399  Recall: 0.5873  F1: 0.6912

Best model saved! (Validation F1 = 0.6912)
Epoch 3/10


Training:   0%|          | 0/2118 [00:00<?, ?it/s]

Validation:   0%|          | 0/537 [00:00<?, ?it/s]


TRAIN -- Loss: 0.4644  Accuracy: 0.7855  Precision: 0.7907  Recall: 0.6881  F1: 0.7359
VAL   -- Loss: 0.4792  Accuracy: 0.7817  Precision: 0.8445  Recall: 0.6355  F1: 0.7252

Best model saved! (Validation F1 = 0.7252)
Epoch 4/10


Training:   0%|          | 0/2118 [00:00<?, ?it/s]